# Đánh Giá Hệ Thống Acne Advisor AI

Notebook này dùng để đánh giá RAG runtime bằng bộ 300 câu hỏi mẫu và các metric deterministic dễ giải thích. Mặc định notebook **không gọi API live**. Khi cần đánh giá thật, chạy backend local rồi đặt `RUN_LIVE_EVAL=True`.

Output chính nằm trong `reports/evaluation/<timestamp>/` gồm `results.csv`, `summary_metrics.json`, `summary_report.md` và thư mục `plots/`.


## 1. Cấu hình chạy

- `RUN_LIVE_EVAL=False`: không gọi `/chat`, chỉ đọc kết quả đã lưu nếu có.
- `RUN_LIVE_EVAL=True`: gọi backend local `/chat`.
- `QUESTION_LIMIT=300`: chạy toàn bộ bộ câu hỏi.
- Notebook không yêu cầu chọn model/provider/judge provider; backend tự dùng default runtime.


In [ ]:
from pathlib import Path

API_BASE_URL = "http://127.0.0.1:8000"
RUN_LIVE_EVAL = False
QUESTION_LIMIT = 300
REQUEST_TIMEOUT_SECONDS = 90
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.5
USE_SAVED_RESULTS_IF_AVAILABLE = True
SAVE_RAW_RESPONSES = True
DATASET_PATH = "notebooks/eval_data/acne_rag_eval_set.jsonl"
REPORT_ROOT = "reports/evaluation"


## 2. Import helper

Helper nằm ở `notebooks/rag_eval_utils.py`. Các hàm chỉ chấm điểm, ghi report và tạo chart; không gọi API hay LLM.


In [ ]:
import json
import os
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(NOTEBOOKS_DIR))

from rag_eval_utils import (
    build_simple_markdown_report,
    create_evaluation_charts,
    read_jsonl,
    score_case,
    summarize_category_scores,
    summarize_core_metrics,
    top_failure_rows,
    write_csv,
    write_jsonl,
)

load_dotenv(PROJECT_ROOT / ".env", override=False)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report_dir = Path(REPORT_ROOT) / timestamp
report_dir.mkdir(parents=True, exist_ok=True)
plots_dir = report_dir / "plots"
print(f"Report directory: {report_dir}")


## 3. Load bộ câu hỏi 300 câu

Dataset JSONL được load từ `DATASET_PATH` và giới hạn bởi `QUESTION_LIMIT`.

In [ ]:
cases = read_jsonl(DATASET_PATH)
if QUESTION_LIMIT is not None:
    cases = cases[: int(QUESTION_LIMIT)]

case_by_id = {case["id"]: case for case in cases}
categories = sorted({case["category"] for case in cases})

print(f"Loaded {len(cases)} cases from {DATASET_PATH}")
print("Categories:", ", ".join(categories))
for case in cases[:5]:
    print(f"- {case['id']} [{case['category']}]: {case['question']}")


## 4. Kiểm tra backend

Cell này chỉ gọi `GET /health` khi `RUN_LIVE_EVAL=True`. Nếu bạn chỉ muốn đọc kết quả cũ hoặc smoke notebook offline thì không cần backend.


In [ ]:
def check_backend_health() -> dict[str, Any]:
    try:
        response = requests.get(f"{API_BASE_URL.rstrip('/')}/health", timeout=10)
        payload = response.json() if response.headers.get("content-type", "").startswith("application/json") else {}
        return {"ok": response.ok, "status_code": response.status_code, "json": payload}
    except Exception as exc:
        return {"ok": False, "status_code": None, "error": exc.__class__.__name__, "message": str(exc)}


if RUN_LIVE_EVAL:
    health = check_backend_health()
    print(json.dumps(health, ensure_ascii=False, indent=2))
    if not health.get("ok"):
        raise RuntimeError("Backend /health is not OK. Start FastAPI before live evaluation.")
else:
    print("RUN_LIVE_EVAL=False; skip backend health check.")


## 5. Chạy đánh giá qua API hoặc đọc kết quả cũ

Payload `/chat` dùng schema tối giản:

```json
{
  "message": "...",
  "session_id": "eval-...",
  "allow_model_fallback": true,
  "bypass_cache": false
}
```

Notebook không truyền `llm_provider` hoặc `llm_model`, để backend dùng default.


In [ ]:
def latest_saved_raw_responses(root: str | Path) -> Path | None:
    candidates = sorted(Path(root).glob("*/raw_responses.jsonl"), key=lambda path: path.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def call_chat_api(case: dict[str, Any]) -> dict[str, Any]:
    session_id = f"eval-{timestamp}-{case['id']}-{uuid.uuid4().hex[:8]}"
    payload = {
        "message": case["question"],
        "session_id": session_id,
        "allow_model_fallback": True,
        "bypass_cache": False,
    }
    started = time.perf_counter()
    try:
        response = requests.post(
            f"{API_BASE_URL.rstrip('/')}/chat",
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        latency_ms = round((time.perf_counter() - started) * 1000, 3)
        try:
            data = response.json()
        except Exception:
            data = {"text": response.text[:1000]}
        metadata = data.get("metadata", {}) if isinstance(data, dict) else {}
        cache = metadata.get("cache", {}) if isinstance(metadata, dict) else {}
        return {
            "case_id": case["id"],
            "question": case["question"],
            "category": case["category"],
            "ok": response.ok,
            "http_status": response.status_code,
            "latency_ms": latency_ms,
            "answer": data.get("answer", "") if isinstance(data, dict) else "",
            "sources": data.get("sources", []) if isinstance(data, dict) else [],
            "source_metadata": data.get("source_metadata", []) if isinstance(data, dict) else [],
            "provider": metadata.get("provider") if isinstance(metadata, dict) else None,
            "model": metadata.get("model") if isinstance(metadata, dict) else None,
            "cache_hit": bool(cache.get("hit", False)) if isinstance(cache, dict) else False,
            "session_id": data.get("session_id", session_id) if isinstance(data, dict) else session_id,
            "raw_response": data,
        }
    except requests.Timeout as exc:
        return {
            "case_id": case["id"],
            "question": case["question"],
            "category": case["category"],
            "ok": False,
            "http_status": None,
            "latency_ms": round((time.perf_counter() - started) * 1000, 3),
            "answer": "",
            "sources": [],
            "error": "timeout",
            "error_message": str(exc),
            "raw_response": {},
        }
    except Exception as exc:
        return {
            "case_id": case["id"],
            "question": case["question"],
            "category": case["category"],
            "ok": False,
            "http_status": None,
            "latency_ms": round((time.perf_counter() - started) * 1000, 3),
            "answer": "",
            "sources": [],
            "error": exc.__class__.__name__,
            "error_message": str(exc),
            "raw_response": {},
        }


raw_responses: list[dict[str, Any]] = []
if RUN_LIVE_EVAL:
    for index, case in enumerate(cases, start=1):
        print(f"[{index}/{len(cases)}] {case['id']}")
        raw_responses.append(call_chat_api(case))
        time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)
elif USE_SAVED_RESULTS_IF_AVAILABLE:
    saved_path = latest_saved_raw_responses(REPORT_ROOT)
    if saved_path:
        raw_responses = read_jsonl(saved_path)
        print(f"Loaded saved raw responses: {saved_path} ({len(raw_responses)} rows)")
    else:
        print("No saved raw_responses.jsonl found. Set RUN_LIVE_EVAL=True to call the local backend.")
else:
    print("Live eval disabled and saved-result mode disabled.")

if SAVE_RAW_RESPONSES:
    write_jsonl(report_dir / "raw_responses.jsonl", raw_responses)
print(f"Raw responses ready: {len(raw_responses)}")


## 6. Tính các chỉ số chính

Chỉ giữ các metric cần cho báo cáo: success, latency, answer, source, keyword, safety, format, out-of-domain và overall score.


In [ ]:
scored_rows = []
for raw in raw_responses:
    case = case_by_id.get(raw.get("case_id"))
    if not case:
        case = {
            "id": raw.get("case_id"),
            "category": raw.get("category", "unknown"),
            "question": raw.get("question", ""),
            "expected_keywords": [],
            "forbidden_keywords": [],
            "requires_sources": False,
        }
    scored_rows.append(score_case(raw, case))

core_metrics = summarize_core_metrics(scored_rows)
category_rows = summarize_category_scores(scored_rows)
failure_rows = top_failure_rows(scored_rows)

print(json.dumps(core_metrics, ensure_ascii=False, indent=2))


## 7. Vẽ biểu đồ

Các biểu đồ được lưu vào `reports/evaluation/<timestamp>/plots/` và hiển thị inline nếu môi trường notebook hỗ trợ.

In [ ]:
chart_result = create_evaluation_charts(scored_rows, core_metrics, plots_dir)
report_chart_paths = chart_result.get("plots", {}) if chart_result.get("status") == "created" else {}
print(json.dumps(chart_result, ensure_ascii=False, indent=2))

try:
    from IPython.display import Image, display

    if chart_result.get("status") == "created":
        for path in chart_result["plots"].values():
            display(Image(filename=path))
except Exception as exc:
    print(f"Inline chart display skipped: {exc.__class__.__name__}: {exc}")


## 8. Xuất báo cáo

Notebook xuất báo cáo Markdown với tiêu đề **Báo Cáo Đánh Giá Hệ Thống Acne Advisor AI**.

Notebook xuất:

- `results.csv`
- `summary_metrics.json`
- `summary_report.md`
- `plots/*.png`
- `raw_responses.jsonl` nếu `SAVE_RAW_RESPONSES=True`


In [ ]:
config_for_report = {
    "question_count": len(scored_rows),
    "api_base_url": API_BASE_URL,
    "run_live_eval": RUN_LIVE_EVAL,
    "timestamp": timestamp,
}

write_csv(report_dir / "results.csv", scored_rows)
(report_dir / "summary_metrics.json").write_text(
    json.dumps(core_metrics, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)
report_text = build_simple_markdown_report(
    config=config_for_report,
    core_metrics=core_metrics,
    category_rows=category_rows,
    failure_rows=failure_rows,
    chart_paths=report_chart_paths,
)
(report_dir / "summary_report.md").write_text(report_text, encoding="utf-8")

print("Exported:")
for name in ["results.csv", "summary_metrics.json", "summary_report.md"]:
    print("-", report_dir / name)
if SAVE_RAW_RESPONSES:
    print("-", report_dir / "raw_responses.jsonl")
print("-", plots_dir)


## 9. Hướng dẫn đọc kết quả

- Mở `summary_report.md` để lấy bảng chỉ số chính và nhận xét ngắn.
- Dùng `results.csv` để lọc từng case fail.
- Dùng `plots/overall_metrics_bar.png` cho slide tổng quan.
- Dùng `plots/category_scores.png` và `plots/top_failure_categories.png` để tìm nhóm cần cải thiện.
- Nếu notebook không có raw response, hãy chạy backend rồi đặt `RUN_LIVE_EVAL=True`.
